In [27]:
import numpy as np
import pandas as pd
from narwhals import Int64
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, ParameterGrid, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.base import clone

# 1. Load Data

In [34]:
train_df = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index_col=0)

# 2. Preprocess

In [35]:
# Dropping unnecessary columns for modeling
train_date = train_df["event_dt"]
train_df = train_df.drop(["event_dt", "orig_age", "vital_flag", "orig_height", "orig_weight", "BMI", "year", "Zone", "Latitude", "Longitude", "SubjectID"], axis = 1)

# Unfloating zip code
train_df["Zip"] = train_df["Zip"].astype("Int64")

# Defining categorical columns
cat_cols = ["SubjectRace", "Zip", "District"]
# Defining discrete/numerical columns
dis_cols = ["SubjectGender", "SubjectAge", "SubjectHeight", "SubjectWeight", "isLA","VehicleYear", "SubjectHasPhotoID", "hour", "night", "isDriver"]
# Numerical columns we would like to scale
scale_cols = ["SubjectAge", "SubjectHeight", "SubjectWeight", "VehicleYear", "hour"]
# Binary columns
bin_cols = ["SubjectGender", "isLA", "SubjectHasPhotoID", "night", "isDriver"]

# Converting variables into strings
train_df[cat_cols] = train_df[cat_cols].astype(str)

target = "search_occurred"
X_train = train_df.drop([target], axis = 1)[cat_cols + dis_cols]
y_train = train_df[target]

# 3. Define classes and pipeline function

In [36]:
class CatBinner(BaseEstimator, TransformerMixin):
    def __init__(self, cols, min_freq=0.01, fill_value="Other"):
        self.cols = cols
        self.min_freq = min_freq
        self.fill_value = fill_value

    def fit(self, X, y=None):
        self.keep_labels = {}
        for c in self.cols:
            freq = X[c].value_counts(normalize=True)
            self.keep_labels[c] = freq[freq >= self.min_freq].index.tolist()

        return self

    def transform(self, X):
        X_copy = X.copy()
        for c in self.cols:
            labels = self.keep_labels[c]
            X_copy[c] = np.where(X_copy[c].isin(labels) | X_copy[c].isna(), X_copy[c], self.fill_value)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class RoundImputed(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        rounded = X[self.cols].round()
        self.min_ = rounded.min()
        self.max_ = rounded.max()
        return self

    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.cols] = X_copy[self.cols].round().clip(self.min_, self.max_, axis=1)
        return X_copy

    def set_output(self, *, transform=None):
        return self

class OrdEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_cols, inner):
        self.cat_cols = cat_cols
        self.inner = inner

    def _encode(self, X):
        X_enc = X.copy()
        X_enc[self.cat_cols] = self.encoder_.transform(X[self.cat_cols])
        return X_enc

    def _decode(self, X_imp):
        X_dec = X_imp.copy()
        X_dec[self.cat_cols] = self.encoder_.inverse_transform(X_imp[self.cat_cols])
        return X_dec

    def fit_transform(self, X, y=None):
        self.encoder_ = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
        self.encoder_.fit(X[self.cat_cols])
        self.inner_ = clone(self.inner)
        return self._decode(self.inner_.fit_transform(self._encode(X), y))

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        return self._decode(self.inner_.transform(self._encode(X)))

    def set_output(self, *, transform=None):
        return self

def build_pipeline(model, cat_cols, dis_cols, scale_cols):
    imputer = IterativeImputer(estimator=HistGradientBoostingRegressor(max_iter=50, random_state=10), skip_complete=True, max_iter=5, tol=1e-2, random_state=10)

    impute = Pipeline([
        ("imputer", imputer),
        ("round", RoundImputed(cols=cat_cols + dis_cols)),
    ])

    impute.set_output(transform="pandas")

    encode = ColumnTransformer([("encode", OneHotEncoder(handle_unknown="ignore"), cat_cols), ("scale", StandardScaler(), scale_cols)], remainder="passthrough", verbose_feature_names_out=False)

    pipeline = Pipeline([
        ("bin", CatBinner(cols=cat_cols, min_freq=0.01, fill_value="Other")),
        ("imputer", OrdEncoder(cat_cols, impute)),
        ("encode", encode),
        ("model", model)
    ], memory="cache_dir")

    return pipeline

# 3a) Define pipeline

In [37]:
pipeline = build_pipeline(DecisionTreeClassifier(random_state=10), cat_cols, bin_cols, scale_cols)

# 4. Define grid search parameters and run grid search cv
## gini/entropy, max_depth, min_samples_split, min_samples_leaf

In [47]:
param_distributions = {"model__criterion":["gini", "entropy"], "model__max_depth":[None, 3, 6, 10, 20], "model__min_samples_split":randint(2,40), "model__min_samples_leaf":randint(1,40)}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

random_search = RandomizedSearchCV(cv = cv, estimator=pipeline, param_distributions= param_distributions, scoring=["roc_auc", "neg_brier_score", "average_precision"], refit = "average_precision", verbose=2, random_state=10, n_iter=50)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END model__criterion=entropy, model__max_depth=20, model__min_samples_leaf=16, model__min_samples_split=2; total time=   1.0s
[CV] END model__criterion=entropy, model__max_depth=20, model__min_samples_leaf=16, model__min_samples_split=2; total time=   1.0s
[CV] END model__criterion=entropy, model__max_depth=20, model__min_samples_leaf=16, model__min_samples_split=2; total time=   0.9s
[CV] END model__criterion=entropy, model__max_depth=20, model__min_samples_leaf=16, model__min_samples_split=2; total time=   1.0s
[CV] END model__criterion=entropy, model__max_depth=20, model__min_samples_leaf=16, model__min_samples_split=2; total time=   1.0s
[CV] END model__criterion=entropy, model__max_depth=10, model__min_samples_leaf=29, model__min_samples_split=27; total time=   0.7s
[CV] END model__criterion=entropy, model__max_depth=10, model__min_samples_leaf=29, model__min_samples_split=27; total time=   0.6s
[CV] END model__cri

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(memo...m_state=10))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [None, 3, ...], 'model__min_samples_leaf': <scipy.stats....002D444F727A0>, 'model__min_samples_split': <scipy.stats....002D441D43460>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","['roc_auc', 'neg_brier_score', ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None ch

In [48]:
# Save random search results
rand_results = pd.DataFrame(random_search.cv_results_)
rand_results.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\dt_hyp_tuning_rand.csv")

In [49]:
rand_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_model__criterion,param_model__max_depth,param_model__min_samples_leaf,param_model__min_samples_split,params,split0_test_roc_auc,...,std_test_neg_brier_score,rank_test_neg_brier_score,split0_test_average_precision,split1_test_average_precision,split2_test_average_precision,split3_test_average_precision,split4_test_average_precision,mean_test_average_precision,std_test_average_precision,rank_test_average_precision
0,0.875622,0.031231,0.203089,0.015975,entropy,20,16,2,"{'model__criterion': 'entropy', 'model__max_de...",0.656672,...,0.000315,38,0.120494,0.117993,0.110622,0.121210,0.121943,0.118452,0.004135,26
1,0.618276,0.093724,0.229204,0.030024,entropy,10,29,27,"{'model__criterion': 'entropy', 'model__max_de...",0.731999,...,0.000277,6,0.150091,0.151055,0.135448,0.150409,0.147312,0.146863,0.005849,1
2,0.886725,0.088690,0.240182,0.035313,entropy,None,30,10,"{'model__criterion': 'entropy', 'model__max_de...",0.675988,...,0.000432,31,0.130294,0.137778,0.114613,0.132750,0.133197,0.129727,0.007934,17
3,0.822671,0.054492,0.253373,0.027133,entropy,None,37,18,"{'model__criterion': 'entropy', 'model__max_de...",0.694633,...,0.000520,27,0.131689,0.145366,0.119768,0.143819,0.138136,0.135756,0.009329,10
4,0.724690,0.029569,0.238784,0.038987,gini,10,25,35,"{'model__criterion': 'gini', 'model__max_depth...",0.731084,...,0.000141,12,0.139946,0.144049,0.135208,0.142435,0.141048,0.140537,0.002998,7
5,0.793116,0.015436,0.220564,0.011671,gini,20,15,15,"{'model__criterion': 'gini', 'model__max_depth...",0.659615,...,0.000490,37,0.114134,0.127185,0.113555,0.121690,0.118238,0.118960,0.005062,25
6,0.993677,0.526100,0.279984,0.077501,entropy,10,26,15,"{'model__criterion': 'entropy', 'model__max_de...",0.732540,...,0.000268,7,0.148790,0.146695,0.135179,0.150220,0.148543,0.145886,0.005469,2
7,0.930465,0.098871,0.534680,0.433669,gini,3,13,3,"{'model__criterion': 'gini', 'model__max_depth...",0.685171,...,0.000132,15,0.098604,0.096329,0.091135,0.100124,0.100299,0.097298,0.003395,37
8,1.114791,0.217698,0.538463,0.192392,entropy,3,37,29,"{'model__criterion': 'entropy', 'model__max_de...",0.690911,...,0.000141,21,0.097923,0.096243,0.091274,0.100014,0.100115,0.097114,0.003254,43
9,2.277160,0.244295,0.580031,0.199786,gini,None,12,30,"{'model__criterion': 'gini', 'model__max_depth...",0.641216,...,0.000273,42,0.108113,0.119538,0.107186,0.110159,0.114048,0.111809,0.004528,31


In [50]:
# Best parameters based on:
## Balanced accuracy
## Precision
print("PRAUC/Average Precision:", rand_results["params"][rand_results["rank_test_average_precision"] == 1].head(1).item())
print("AUC:", rand_results["params"][rand_results["rank_test_roc_auc"] == 1].head(1).item())
## Brier
print("Brier score:", rand_results["params"][rand_results["rank_test_neg_brier_score"] == 1].head(1).item())

PRAUC/Average Precision: {'model__criterion': 'entropy', 'model__max_depth': 10, 'model__min_samples_leaf': 29, 'model__min_samples_split': 27}
AUC: {'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 2, 'model__min_samples_split': 15}
Brier score: {'model__criterion': 'entropy', 'model__max_depth': 6, 'model__min_samples_leaf': 6, 'model__min_samples_split': 20}


## 4a) Use grid search to more closely examine optimal parameters, using random search results

In [51]:
# Since we decided to focus on PR-AUC/Average Precision, we will closely examine parameters around optimal parameters found for PRAUC in random search
param_grid = {"model__criterion":["entropy"], "model__max_depth":[9, 10, 11, 12], "model__min_samples_split":[26, 27, 28, 29], "model__min_samples_leaf":[28, 29, 30, 31]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

grid_search = GridSearchCV(cv = cv, estimator=pipeline, param_grid= param_grid, scoring=["roc_auc", "neg_brier_score", "average_precision"], refit = "average_precision", verbose=2)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 64 candidates, totalling 320 fits
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=26; total time=   0.9s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=26; total time=   0.9s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=26; total time=   0.8s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=26; total time=   0.9s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=26; total time=   0.8s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=27; total time=   0.6s
[CV] END model__criterion=entropy, model__max_depth=9, model__min_samples_leaf=28, model__min_samples_split=27; total time=   0.9s
[CV] END model__crite

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(memo...m_state=10))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['entropy'], 'model__max_depth': [9, 10, ...], 'model__min_samples_leaf': [28, 29, ...], 'model__min_samples_split': [26, 27, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","['roc_auc', 'neg_brier_score', ...]"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'average_precision'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary

In [52]:
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\modeling_results\dt_hyp_tuning_grid.csv")

In [54]:
# Best parameters based on:
print("PRAUC/Average Precision:", grid_results["params"][grid_results["rank_test_average_precision"] == 1].head(1).item())
print("AUC:", grid_results["params"][grid_results["rank_test_roc_auc"] == 1].head(1).item())
## Brier
print("Brier score:", grid_results["params"][grid_results["rank_test_neg_brier_score"] == 1].head(1).item())

PRAUC/Average Precision: {'model__criterion': 'entropy', 'model__max_depth': 11, 'model__min_samples_leaf': 31, 'model__min_samples_split': 26}
AUC: {'model__criterion': 'entropy', 'model__max_depth': 9, 'model__min_samples_leaf': 31, 'model__min_samples_split': 26}
Brier score: {'model__criterion': 'entropy', 'model__max_depth': 9, 'model__min_samples_leaf': 31, 'model__min_samples_split': 26}


In [56]:
# Implement one-standard-error rule from Elements of Statistical Learning
# Find index of obs with highest mean test average PR-AUC
ind_opt = grid_results["mean_test_average_precision"].idxmax()
# Find according mean and sd
mean_opt = grid_results["mean_test_average_precision"].loc[ind_opt]
se_opt = grid_results["std_test_average_precision"].loc[ind_opt]

# We want to minimize complexity; pick smallest depth, largest min samples leaf, and largest min samples split
opt_depth = min(grid_results["param_model__max_depth"][(mean_opt - se_opt <= grid_results["mean_test_average_precision"]) & (grid_results["mean_test_average_precision"] <= mean_opt + se_opt)])
opt_min_samples_leaf = max(grid_results["param_model__min_samples_leaf"][(mean_opt - se_opt <= grid_results["mean_test_average_precision"]) & (grid_results["mean_test_average_precision"] <= mean_opt + se_opt)])
opt_min_samples_split = max(grid_results["param_model__min_samples_split"][(mean_opt - se_opt <= grid_results["mean_test_average_precision"]) & (grid_results["mean_test_average_precision"] <= mean_opt + se_opt)])

print(f"Optimal depth:", opt_depth, "Optimal min samples leaf:", opt_min_samples_leaf, "Optimal min samples split:", opt_min_samples_split)

Optimal depth: 9 Optimal min samples leaf: 31 Optimal min samples split: 29


## We use PR-AUC/"Average Precision" to determine the optimal parameters for the decision tree. We first use randomsearch over a large range of potential parameters, and narrow it down with gridsearch. We then implement the one standard rule to find the least complex model that is within 1SD of the highest mean test PR-AUC. This reduces the risk of overfitting, and makes running our model more computationally efficient while maintaining generally optimal performance. Based on this, our optimal parameters are criterion = "entropy", max_depth = 9, min_samples_leaf = 31, min_samples_split = 29